In [1]:
from utils.f_0_dirs import get_data_dirs
from f_3a_parse_linear import parse_linear
from f_3b_parse_dyn import parse_dyn
from f_3_model_to_tex import generate_latex_table
import json
dirs = get_data_dirs(segment="model")

In [5]:
run_type = 'dyn' # 'test' or 'full

if run_type == 'full':
    linear_file_names = [
        ("results_0_tfp_combined", 'itx'),
        ("results_1a_lmm_basic_str", 'struct'),
        ("results_1b_lmm_ols", 'matrix', None, None, False, ['r2-adj', 'r2-overall-adj']),
        ("results_1c_lmm_3r_str", 'struct'),
        ("results_2a_dd_basic_str", 'struct', True),
        ("results_2b_dd_noz_str", 'struct', 6, True),
        ("results_2c_dd_ols_str", 'matrix', None, None, False, ['r2-adj', 'r2-overall-adj'])
    ]
    for tup in linear_file_names:
        while len(tup) < 6:
            tup += (None,)
        in_file_name, rename, start_at, hide_fe, stars, r2_type = tup
        lin0_data = parse_linear(dirs.output_dir / f"{in_file_name}.txt")
        generate_latex_table(lin0_data, dirs.output_dir / f"{in_file_name}.tex", rename_strat=rename, start_at=start_at, hide_fe=hide_fe, stars=stars, r2_type=r2_type)

if run_type == 'test':
    in_file_name = "results_1a_lmm_basic_str"
    lin0_data = parse_linear(dirs.output_dir / f"{in_file_name}.txt")
    json_data = json.dumps(lin0_data, indent=4)
    print(json_data)
    generate_latex_table(lin0_data, dirs.output_dir / f"{in_file_name}.tex", stars=False)

if run_type == 'dyn' or run_type == 'full':
    in_file_names = [
        ("results_3a_basic_str", 'struct'),
        # "instr_3a_identif"
        # "instr_3b_Jover",
        # "instr_3c_ar"
    ]
    for tup in in_file_names:
        while len(tup) < 6:
            tup += (None,)
        in_file_name, rename, start_at, hide_fe, stars, r2_type = tup
        lin0_data = parse_dyn(dirs.output_dir / f"{in_file_name}.txt")
        generate_latex_table(lin0_data, dirs.output_dir / f"{in_file_name}.tex", rename_strat=rename, start_at=start_at, hide_fe=hide_fe, stars=stars, r2_type=r2_type)

    # in_file_name = "instr_3a_dyn"
    # lin0_data = parse_dyn(dirs.output_dir / f"{in_file_name}.txt")
    # generate_latex_table(lin0_data, dirs.output_dir / f"{in_file_name}.tex")

Successfully generated LaTeX table with 2 models at C:\Users\lazyst\Files\ucl\Dissertation\model\output\results_3a_basic_str.tex


### Output current working panel table to a parquet file
So we can use it in a lower python version, which ibis doesn't support

In [1]:
import ibis
from utils.f_0_dirs import get_data_dirs

dirs = get_data_dirs(segment="model")
con = ibis.duckdb.connect(dirs.db_path, read_only=True)
table_panel_name = "working_yearly_n"

# Export to parquet
df_panel = con.table(table_panel_name).execute()
df_panel.to_parquet(dirs.tmp_dir / f"{table_panel_name}.parquet", index=False)
df_panel.to_csv(dirs.tmp_dir / f"{table_panel_name}.csv", index=False)

### Export CSV to latex

In [ ]:
import pandas as pd
from utils.f_0_dirs import get_data_dirs
from model.src.f_3_model_to_tex import generate_latex_from_generic

d_dirs = get_data_dirs(segment="descriptives")
filename = "tse_ic_selection"

with open(d_dirs.output_dir / f"{filename}.csv", "r") as f:
    df_input = pd.read_csv(f)
    generate_latex_from_generic(df_input, d_dirs.output_dir / f"{filename}.tex", format_float=",0.2f")

Successfully exported 4 rows to /mnt/c/Users/lazym/Documents/Code/dissertation/descriptives/output/tse_ic_selection.csv.tex


In [ ]:
import pandas as pd
from utils.f_0_dirs import get_data_dirs
from model.src.f_3_model_to_tex import generate_latex_from_generic

d_dirs = get_data_dirs(segment="calculations")
filename = "groupWprefixes"
outname = "3_group_w_prefixes"

with open(d_dirs.input_dir / f"{filename}.csv", "r") as f:
    df_input = pd.read_csv(f)

    # In column 2, replace "W" with "\mathbf W" and replace I with "\mathbb I"
    df_input.iloc[:, 1] = df_input.iloc[:, 1].str.replace("V", "\\mathbf V", regex=False)
    df_input.iloc[:, 1] = df_input.iloc[:, 1].str.replace("W", "\\mathbf W", regex=False)
    df_input.iloc[:, 1] = df_input.iloc[:, 1].str.replace("I", "\\mathbb I", regex=False)

    # Drop the 3rd row (including header) and the 3rd column (index 2)
    df_input = df_input.drop(df_input.columns[2], axis=1)
    df_input = df_input.drop(df_input.index[1])

    generate_latex_from_generic(df_input, d_dirs.output_dir / f"{outname}.tex", rename_strat="none", hrows=2, hcols=2)

Successfully exported df (26x5) to C:\Users\lazyst\Files\ucl\Dissertation\calculations\output\3_group_w_prefixes.tex
